# Project 3: Production Data Quality Gate and Automated Audit Pipeline

**Module**: 13_end_to_end_projects  
**Notebook**: 03_data_quality_pipeline.ipynb  
**Environment**: Apache Spark 4.2.1 · Standalone Cluster · PySpark · Python 3.13 · Java 21  

---

## Learning Objectives

- **What will I learn?**: Building an automated data quality pipeline: schema conformance, null auditing, uniqueness verification, range checks, foreign key integrity, and generating an automated data health scorecard.
- **Why does this topic exist?**: Garbage in, garbage out. Production data platforms must halt bad data at ingestion gates.
- **What problem does it solve?**: Handling corrupt records in /opt/spark-data/quality/orders_dirty.csv without crashing the job.
- **Where is it used in real-world Data Engineering?**: Implementing Great Expectations / Deequ style data quality gates.

---

### 1. Initialize SparkSession on Standalone Cluster

We connect to `spark://spark-master:7077` with the driver configured to bind to container hostname `jupyter`.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
import time

spark = (
    SparkSession.builder
    .appName("Data_Quality_Project")
    .master("spark://spark-master:7077")
    .config("spark.driver.host", "jupyter")
    .config("spark.driver.bindAddress", "0.0.0.0")
    .config("spark.eventLog.enabled", "true")
    .config("spark.eventLog.dir", "file:///opt/spark-events")
    .config("spark.sql.warehouse.dir", "file:///opt/spark-data/warehouse")
    .config("spark.hadoop.fs.permissions.umask-mode", "000")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

print(f" SparkSession initialized successfully!")
print(f"Spark Version:      {spark.version}")
print(f"Master URL:         {spark.sparkContext.master}")
print(f"App Name:           {spark.sparkContext.appName}")
print(f"Default Parallelism: {spark.sparkContext.defaultParallelism}")
print(f"Spark UI available at: http://localhost:4040")

## 3. Reading Dirty Data and Defining Quality Constraints

In [ ]:
dirty_df = spark.read.option("header", True).option("inferSchema", True).csv("/opt/spark-data/quality/orders_dirty.csv")
print(f"Total input rows in audit pool: {dirty_df.count():,}")
dirty_df.show(3)

## 4. Rule-Based Record Categorization (Valid vs Quarantined)

In [ ]:
# Define quality checks
# Rule 1: order_id is not null and positive
# Rule 2: customer_id is not null
# Rule 3: quantity > 0 and unit_price > 0
# Rule 4: discount between 0.0 and 1.0

flagged_df = (
    dirty_df
    .withColumn("is_null_id", F.col("order_id").isNull() | F.col("customer_id").isNull())
    .withColumn("is_invalid_qty", F.col("quantity").isNull() | (F.col("quantity") <= 0))
    .withColumn("is_invalid_price", F.col("unit_price").isNull() | (F.col("unit_price") <= 0))
    .withColumn("is_invalid_discount", F.col("discount").isNull() | (F.col("discount") < 0.0) | (F.col("discount") > 1.0))
    .withColumn("is_valid", ~(F.col("is_null_id") | F.col("is_invalid_qty") | F.col("is_invalid_price") | F.col("is_invalid_discount")))
)

valid_records = flagged_df.filter("is_valid = True").drop("is_null_id", "is_invalid_qty", "is_invalid_price", "is_invalid_discount", "is_valid")
quarantine_records = flagged_df.filter("is_valid = False")

print(f"Valid Clean Records:     {valid_records.count():,}")
print(f"Quarantined Bad Records: {quarantine_records.count():,}")

---

## Experiment: Generating Automated Data Quality Scorecard

- **1. What are we testing?**: Aggregate metrics across all quality rules into an executive summary report.
- **2. What do we expect?**: A structured DataFrame reporting error counts and pass percentages per rule.

In [ ]:
total_rows = dirty_df.count()

scorecard = flagged_df.select(
    F.lit(total_rows).alias("total_records"),
    F.sum(F.col("is_null_id").cast("int")).alias("failed_null_ids"),
    F.sum(F.col("is_invalid_qty").cast("int")).alias("failed_quantity_check"),
    F.sum(F.col("is_invalid_price").cast("int")).alias("failed_price_check"),
    F.sum(F.col("is_invalid_discount").cast("int")).alias("failed_discount_check"),
    F.sum(F.col("is_valid").cast("int")).alias("passed_records")
)

print("=== DATA QUALITY SCORECARD ===")
scorecard.show(vertical=True)

**3. What happened?**: Generated complete data quality audit report.  
**4. Why did it happen?**: Aggregated rule flags into summary counters.  
**5. Practical Engineering Lesson**: Always publish data quality scorecards to monitoring systems (Datadog/Prometheus) to alert on ingestion anomalies.

---

## Practical Exercises

### Exercise 1 (Beginner): Write Quarantine Table
**Problem Statement**: Write quarantined records to /opt/spark-data/warehouse/quarantine_orders.  
**Input / Context**: quarantine_records

In [ ]:
# Write your solution for Exercise 1 here

### Exercise 2 (Intermediate): Referential Integrity Check
**Problem Statement**: Check if customer_ids in dirty_df exist in customers.csv.  
**Input / Context**: Provided dataset

In [ ]:
# Write your solution for Exercise 2 here

### Exercise 3 (Advanced): Automated Quality Threshold Gate
**Problem Statement**: Raise an exception if pass rate is below 90%.  
**Input / Context**: Provided dataset

In [ ]:
# Write your solution for Exercise 3 here

---

## Reference Solutions

<details>
<summary><b>Click to expand full solutions</b></summary>

#### Solution 1 (Write Quarantine Table):
```python
quarantine_records.write.mode('overwrite').parquet('/opt/spark-data/warehouse/quarantine_orders')
```

#### Solution 2 (Referential Integrity Check):
```python
custs = spark.read.option('header', True).csv('/opt/spark-data/retail/customers.csv')
orphan_orders = dirty_df.join(custs,'customer_id','left_anti')
print('Orphan orders count:', orphan_orders.count())
```

#### Solution 3 (Automated Quality Threshold Gate):
```python
pass_rate = valid_records.count() / total_rows
assert pass_rate >= 0.90, f'Quality threshold failed! Pass rate: {pass_rate:.2%}'
print('Quality gate passed!')
```

</details>

---

## Module Summary & Interview Guide

### Key Takeaways
- **Quality Gates**: Quarantine bad rows without crashing the pipeline.
- **Scorecarding**: Quantify data quality metrics across batch runs.

### Essential Technical Terms
- `Quarantine`: Isolating corrupt records into a separate dead-letter queue table.
- `Data Quality Gate`: Automated threshold validation step.

### Common Engineering Mistakes
- **Silently dropping bad data without audit**: Hides upstream data ingestion bugs.

### Interview Questions on This Topic
- **Q: How do you handle bad records without failing an entire batch job?**
  - *A*: Flag records using boolean rule expressions, split the DataFrame into valid and quarantine subsets, write valid rows to the lake, and write quarantine rows to a dead-letter queue with audit metadata.

### What to Study Next
Open 04_streaming_analytics_pipeline.ipynb for real-time event analytics.

---

### Tear Down SparkSession
Always close your session when finishing to release executor threads and driver ports.

In [ ]:
spark.stop()
print('SparkSession successfully stopped.')